# Day 21: Date Parsing and Datetime Basics

**Dataset:** `Superstore_Dataset.csv`

This notebook converts text-based order and shipping dates into Pandas datetime values, checks date validity, extracts calendar information, and uses dates to analyze Superstore orders.

## 1. Import Libraries and Load the Dataset

The dates are currently stored as text. Start by loading the dataset and checking the original values and data types.

In [2]:
import pandas as pd

# Load the dataset
df = pd.read_csv("../datasets/Superstore_Dataset.csv", encoding="cp1252")

# Preview the date columns and their current types
df[["Order ID", "Order Date", "Ship Date", "Sales"]].head()


,Order ID,Order Date,Ship Date,Sales
0,CA-2016-152156,11/8/2016,11/11/2016,261.9600
1,CA-2016-152156,11/8/2016,11/11/2016,731.9400
2,CA-2016-138688,6/12/2016,6/16/2016,14.6200
3,US-2015-108966,10/11/2015,10/18/2015,957.5775
4,US-2015-108966,10/11/2015,10/18/2015,22.3680


## 2. Convert Text to Datetime

`pd.to_datetime()` converts date strings into Pandas datetime values. Supplying the format makes the month/day/year pattern explicit; `errors="coerce"` turns values that do not match into `NaT` instead of stopping the conversion.

In [3]:
date_format = "%m/%d/%Y"

df["Order Date"] = pd.to_datetime(df["Order Date"], format=date_format, errors="coerce")
df["Ship Date"] = pd.to_datetime(df["Ship Date"], format=date_format, errors="coerce")

print("Order Date dtype:", df["Order Date"].dtype)
print("Ship Date dtype:", df["Ship Date"].dtype)
df[["Order Date", "Ship Date"]].head()


Order Date dtype: datetime64[us]
Ship Date dtype: datetime64[us]


,Order Date,Ship Date
0,2016-11-08,2016-11-11
1,2016-11-08,2016-11-11
2,2016-06-12,2016-06-16
3,2015-10-11,2015-10-18
4,2015-10-11,2015-10-18


## 3. Extract Calendar Information

Once a column has a datetime dtype, the `.dt` accessor exposes useful parts of each date. Add the order year, month, day, and weekday, then calculate the number of days between ordering and shipping.

In [4]:
df["Order Year"] = df["Order Date"].dt.year
df["Order Month"] = df["Order Date"].dt.month
df["Order Day"] = df["Order Date"].dt.day
df["Order Weekday"] = df["Order Date"].dt.day_name()
df["Shipping Days"] = (df["Ship Date"] - df["Order Date"]).dt.days

df[["Order Date", "Order Year", "Order Month", "Order Day", "Order Weekday", "Ship Date", "Shipping Days"]].head()


,Order Date,Order Year,Order Month,Order Day,Order Weekday,Ship Date,Shipping Days
0,2016-11-08,2016,11,8,Tuesday,2016-11-11,3
1,2016-11-08,2016,11,8,Tuesday,2016-11-11,3
2,2016-06-12,2016,6,12,Sunday,2016-06-16,4
3,2015-10-11,2015,10,11,Sunday,2015-10-18,7
4,2015-10-11,2015,10,11,Sunday,2015-10-18,7


## 4. Check Date Quality

`errors="coerce"` makes invalid or missing date values visible as `NaT`. Check for those values and for shipping dates that occur before the corresponding order date.

In [5]:
invalid_date_rows = df[df[["Order Date", "Ship Date"]].isna().any(axis=1)]
ship_before_order = df["Ship Date"] < df["Order Date"]

quality_summary = pd.DataFrame({
    "check": [
        "Missing or invalid order/ship date",
        "Ship date before order date"
    ],
    "rows_failed": [
        len(invalid_date_rows),
        int(ship_before_order.sum())
    ]
})

quality_summary


,check,rows_failed
0,Missing or invalid order/ship date,0
1,Ship date before order date,0


## 5. Analyze Sales by Date

Group sales by order year and month to compare periods. For weekday counts, count unique order IDs so an order with several product rows is not counted several times.

In [6]:
annual_sales = df.groupby("Order Year")["Sales"].sum().round(2)
monthly_sales = df.groupby(df["Order Date"].dt.to_period("M"))["Sales"].sum().sort_values(ascending=False)
weekday_order_counts = (
    df.groupby("Order Weekday")["Order ID"]
    .nunique()
    .reindex(["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"])
    .fillna(0)
    .astype(int)
)

print("Sales by order year:")
print(annual_sales.to_string())
print("\nFive highest-sales months:")
print(monthly_sales.head(5).round(2).to_string())
print("\nUnique orders by weekday:")
print(weekday_order_counts.to_string())
print("\nAverage shipping time:", round(df["Shipping Days"].mean(), 2), "days")


Sales by order year:
Order Year
2014    484247.50
2015    470532.51
2016    609205.60
2017    733215.26

Five highest-sales months:
Order Date
2017-11    118447.82
2016-12     96999.04
2017-09     87866.65
2017-12     83829.32
2014-09     81777.35
Freq: M

Unique orders by weekday:
Order Weekday
Monday       920
Tuesday      558
Wednesday    182
Thursday     746
Friday       916
Saturday     837
Sunday       850

Average shipping time: 3.96 days


## 6. Final Datetime Dataset

The original date columns are now datetime values, and the new calendar and shipping-duration columns are ready for later filtering, grouping, or visualization.

In [7]:
df[[
    "Order ID",
    "Order Date",
    "Ship Date",
    "Order Year",
    "Order Month",
    "Order Day",
    "Order Weekday",
    "Shipping Days"
]].head()


,Order ID,Order Date,Ship Date,Order Year,Order Month,Order Day,Order Weekday,Shipping Days
0,CA-2016-152156,2016-11-08,2016-11-11,2016,11,8,Tuesday,3
1,CA-2016-152156,2016-11-08,2016-11-11,2016,11,8,Tuesday,3
2,CA-2016-138688,2016-06-12,2016-06-16,2016,6,12,Sunday,4
3,US-2015-108966,2015-10-11,2015-10-18,2015,10,11,Sunday,7
4,US-2015-108966,2015-10-11,2015-10-18,2015,10,11,Sunday,7


## Outcome

Using `Superstore_Dataset.csv`, I converted `Order Date` and `Ship Date` from text into datetime columns, extracted order year, month, day, and weekday, and calculated shipping duration. Both date-quality checks passed with zero failures. The date-based analysis showed that 2017 had the highest total sales, November 2017 was the strongest month in the dataset, and average shipping time was about 3.96 days. These results show how parsed dates enable reliable filtering, validation, and time-based analysis.